In [ ]:
import os
os.environ.pop("TRITON_PTXAS_PATH", None)

In [ ]:
# import soundfile as sf
from vllm import LLM, SamplingParams
import librosa
import numpy as np
# 1) 讀兩個 wav（sf.read 回傳 (time, channels) 或 (time,)；vLLM 會自動處理 stereo->mono 等）
llm = LLM(
    model="Qwen/Qwen2.5-Omni-7B",
    max_model_len=30000,
    max_num_seqs=1,
    limit_mm_per_prompt={"audio": 4},  # 只寫 audio
    trust_remote_code=True,
    # enforce_eager=True,
)

In [ ]:
default_system = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
    "capable of perceiving auditory and visual inputs, as well as generating text and speech."
)

# 3) 你的問題（把兩段音訊都放進同一個 user message）

sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=256,  # 對應你原本 max_new_tokens=256
)

question = (
    "Introduce youself, qwen 2.5"
)

prompt = (
    f"<|im_start|>system\n{default_system}<|im_end|>\n"
    "<|im_start|>user\n"
    f"{question}<|im_end|>\n"
    "<|im_start|>assistant\n"
)

outputs = llm.generate(
    [
        {
            "prompt": prompt,
            # # 這個案例不是 video，所以不需要 use_audio_in_video
            # "mm_processor_kwargs": {"use_audio_in_video": False},
        }
    ],
    sampling_params=sampling_params,
)


print('output:')
print(outputs[0].outputs[0].text)

In [ ]:
def load_audio(path, sr=16000):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return (y.astype(np.float32), sr)

    

In [ ]:
sample_problem = {'id': 'VCTK-Corpus/wav48/p225/p225_026.wav',
 'dataset': 'VCTK-Corpus',
 'seed_description': '[00:00-00:03] How are you, sir ? (Gender: Female, Pitch:high, Accent: us, Age: 23, Emotion: neutral)',
 'prompt': 'How would you describe the mood in 5 words or fewer?',
 'response': 'Friendly and casual.',
 'messages': [{'role': 'system',
   'content': 'Imagine you can **hear** the audio clips. The audio clips are wrapped between <start_audio> and <end_audio>.\nFocus on the audios and respond directly to the prompts.'},
  {'role': 'user',
   'content': '<start_audio><|AUDIO|><end_audio>\n\nHow would you describe the mood in 5 words or fewer?',
   'audios': [{'audio': 'VCTK-Corpus/wav48/p225/p225_026.wav',
     'text': 'How are you, sir ?',
     'seed_description': '[00:00-00:03] How are you, sir ? (Gender: Female, Pitch:high, Accent: us, Age: 23, Emotion: neutral)'}]}]}

In [ ]:
sample_path = '/home/u1501463/tool_use_LALM/p225_026_mic2.flac'
audio1, sr1 = load_audio(sample_path)

In [ ]:
def QA_pair_gen_prompt(n:int, audio_token, description, labels, language):

    TOOL_COUNT_HINTS = {
        1: "Single-tool questions. Examples: transcribe the whole audio, separate vocals only, clip a specific moment.",
        2: "Two-step questions. Examples: clip a segment then transcribe it; separate vocals then transcribe them; clip two different segments for comparison.",
        3: "Three-step chains. Examples: clip → separate → transcribe; or clip two segments (2 calls) then compare their transcripts (1 ASR call).",
        4: "Complex multi-step. Examples: clip → separate → ASR on one stem → ASR on another stem.",
    }

    tool_count_hint = TOOL_COUNT_HINTS[n]

    tool_definitions = '''The LALM has access to the following tools:

- **clipping**: Extract a time segment from an audio file.
Parameters: audio_id, audio_begin (HH:MM:SS.mmm), audio_end (HH:MM:SS.mmm)
Returns: a new clip_id and an <audio> token representing the clipped audio.

- **source_separation**: Separate an audio file into individual sound sources based on provided labels. Powered by SAM-Audio, which accepts any audio type (speech, music, environmental sounds, etc.) and separates sources according to the labels you specify.
Parameters:
- audio_id (string)
- audio_begin (HH:MM:SS.mmm)
- audio_end (HH:MM:SS.mmm)
- stems (array of strings): labels describing the sound sources to separate,
    e.g. ["speaker_1", "background_noise"], ["piano", "vocals"],
    ["engine_sound", "crowd"], ["dog_barking", "music"].
    The labels should be grounded in the audio metadata provided.


- **asr**: Transcribe speech from audio to text.
Parameters: audio_id, audio_begin, audio_end, language (optional BCP-47)
Returns: timestamped transcript with speaker_id and confidence.
'''

    prompt_template =\
f"""You are a data generation assistant. Your task is to create a question-answer pair that requires exactly {n} tool call(s) to solve, based on the provided audio and its metadata.

---

## Audio Information

Audio: {audio_token}
Description: {description}
Labels: {labels}

---

## Available Tools

{tool_definitions}

---

## Your Task

Generate ONE question-answer pair that satisfies ALL of the following requirements:

### Question Requirements
1. The question MUST be answerable using the provided audio — do not invent information that is not present in the audio.
2. The question MUST require exactly {n} tool call(s) to answer. No more, no less.
3. The question MUST have a single correct, verifiable answer. Do NOT generate open-ended, subjective, or opinion-based questions.
4. The answer MUST be fully determined by the audio content and the tool results — not by the description or labels alone.
5. The question should be natural and concise, as if asked by a real user.

### Reasoning Requirements
1. Provide a step-by-step reasoning chain that shows how to arrive at the answer.
2. For each tool call, clearly state: why this tool is needed, what parameters to use, and what result is expected.
3. Each tool call must use the correct format defined in the Tool Invocation section below.
4. The reasoning must be consistent with the labels and description provided.

### Answer Requirements
1. The answer must be short, factual, and directly derived from the tool results.
2. Do NOT include reasoning or explanation in the answer field.

---

## Tool Invocation Format

When specifying a tool call in the reasoning chain, use the following format:

```json
{{
  "tool_call": {{
    "name": "<tool_name>",
    "request_id": "<unique_id>",
    "parameters": {{ ... }}
  }}
}}
```

Tool result placeholder format:
<tool_result>
[TOOL_RESULT_BEGIN]
request_id: <matching_id>
tool: <tool_name>
status: success
<expected result based on labels and description>
[TOOL_RESULT_END]
</tool_result>

---

## Output Format

You MUST output exactly three clearly separated blocks. Do not merge them.

### Block 1 — Question
[QUESTION]
<your question here>
[/QUESTION]

### Block 2 — Reasoning and Tool Use
[REASONING]
Step 1: <explain what information is needed and why>
Tool call:
<tool_call json>
Expected tool result:
<tool_result block>
Step 2: <interpret the tool result and continue reasoning>
...
(repeat for each tool call)
Final reasoning: <derive the answer from all tool results>
[/REASONING]

### Block 3 — Answer
[ANSWER]
<short factual answer>
[/ANSWER]

---

## Self-Check Before Outputting

Before generating your final output, verify the following:

- [ ] The question cannot be answered without invoking the tools.
- [ ] Exactly {n} tool call(s) appear in the reasoning.
- [ ] The answer is uniquely determined — no ambiguity, no subjectivity.
- [ ] Every tool parameter (audio_id, timestamps, etc.) is consistent with the provided audio information.
- [ ] The answer matches what the tool results and reasoning actually conclude.
- [ ] The [QUESTION], [REASONING], and [ANSWER] blocks are fully separated with no cross-contamination.

If any check fails, revise before outputting.
"""

    return prompt_template

In [ ]:
# from train_QA_gen_prompt import QA_pair_gen_prompt
ppt = QA_pair_gen_prompt(
    n = 1,
    audio_token="<|audio_bos|><|AUDIO|><|audio_eos|>\n",
    description='[00:00-00:03] How are you, sir ? (Gender: Female, Pitch:high, Accent: us, Age: 23, Emotion: neutral)',
    labels = None,
    language = 'english'
)

In [ ]:
print(ppt)

In [ ]:
prompt = (
    f"<|im_start|>system\n{default_system}<|im_end|>\n"
    "<|im_start|>user\n"
    f"{ppt}"
    f"{question}<|im_end|>\n"
    "<|im_start|>assistant\n"
)

outputs = llm.generate(
    [
        {
            "prompt": prompt,
            "multi_modal_data": {
                "audio": [
                    (audio1, sr1),
                ],
            },
        }
    ],
    sampling_params= SamplingParams(
        temperature=0.5,
        max_tokens=256,  # 對應你原本 max_new_tokens=256
    )
)

print('output:')
print(outputs[0].outputs[0].text)

In [ ]:
print(outputs[0].outputs[0].text)

[QUESTION]What is the gender of the speaker in the audio?，/QUESTION，[REASONING]Step 1: To answer the question about the gender of the speaker, we need to use the information provided in the audio metadata. The metadata directly states the gender of the speaker as female.，/REASONING，Step 2: No tool call is needed in this case as the answer is directly provided in the metadata.，/REASONING，Final reasoning: The gender of the speaker is female as stated in the audio metadata.，/REASONING，[ANSWER]Female.，/ANSWER，If you have any other questions or need further clarification, feel free to ask.

In [ ]:
import json
qa_pair = outputs[0].outputs[0].text
qa_pair = json.loads(qa_pair)

In [ ]:
qa_pair

In [ ]:
validator_prompt_template = """\
You are a critical evaluator for training data quality of a Large Audio Language Model (LALM).

Your task is to evaluate whether the tool chain used in the given Q/A pair is reasonable and necessary for answering the question.

---

## Available Tools and Their Capabilities

- **clipping**: Extracts a time segment from audio. Use when only a portion of the audio is relevant.
  CANNOT determine: speaker identity, gender, emotion, language, transcription, or any semantic content.

- **source_separation**: Separates audio into stems (vocals / drums / bass / guitar / piano / other).
  CAN help with: isolating a specific sound source before further analysis.
  CANNOT determine: speaker identity, gender, emotion, transcription, or any semantic content directly.

- **asr**: Transcribes speech to text.
  CAN help with: understanding spoken content, identifying spoken language, analyzing what was said.
  CANNOT determine: speaker gender, speaker identity, non-speech sound events, music genre, or audio quality.

---

## Q/A Pair to Evaluate

### Question
{question}

### Reasoning
{reasoning}

### Tool Chain
```json
{tool_chain}
```

---

## Evaluation Criteria

For each tool call in the chain, judge against ALL of the following:

1. **Necessity**: Is this tool call genuinely required to answer the question?
   - Would skipping this tool make the question unanswerable or significantly harder?

2. **Capability alignment**: Does this tool's output actually contribute useful information toward answering the question?
   - Flag if the tool is incapable of providing what the reasoning claims it provides.

3. **Parameter validity**: Are the parameters consistent with each other and with the question?
   - audio_end > audio_begin
   - Referenced audio_id exists (either the original or returned by a previous tool call)

4. **Chain coherence**: Does each tool call logically follow from the previous result?
   - Is the output of turn N actually used as input or context for turn N+1?

5. **Redundancy**: Is any tool call duplicated or replaceable by another tool already in the chain?

6. **Sufficiency**: Does the final tool result provide enough information to produce the stated final_response?

---

## Output Format

Respond ONLY with a valid JSON object. Do not add any explanation outside the JSON.

```json
{{
  "verdict": "pass | fail",
  "tool_count_expected": {n},
  "tool_count_actual": <number of tool calls in the chain>,
  "evaluations": [
    {{
      "turn": 1,
      "tool_name": "<tool_name>",
      "necessity": "justified | unnecessary | questionable",
      "capability_aligned": true,
      "parameter_valid": true,
      "issues": []
    }}
  ],
  "chain_coherence": "coherent | broken",
  "chain_coherence_reason": "<brief explanation>",
  "redundancy": "none | partial | full",
  "redundancy_reason": "<brief explanation if redundancy is not none>",
  "sufficiency": "sufficient | insufficient",
  "sufficiency_reason": "<brief explanation>",
  "overall_issues": [],
  "suggestion": "<If verdict is fail, describe concisely what should be changed. If pass, leave empty string.>"
}}
```

---

## Evaluation Rules

1. **Be strict on capability alignment**: If a tool is used to infer something it fundamentally cannot provide (e.g. using asr to determine speaker gender, using source_separation to identify language), mark capability_aligned as false and verdict as fail.

2. **Be strict on necessity**: If the question can be answered by fewer tools without loss of correctness, flag the redundant tool calls and set redundancy accordingly.

3. **Chain must be self-consistent**: If turn N produces audio_id X but turn N+1 references a different audio_id without justification, mark chain_coherence as broken.

4. **verdict is pass only if ALL of the following hold**:
   - All tool calls are necessary
   - All tool calls are capability-aligned
   - All parameters are valid
   - Chain is coherent
   - No full redundancy
   - Final result is sufficient to answer the question

5. **Partial redundancy does not automatically fail**: If a redundant step exists but does not break correctness, set redundancy to partial and keep verdict as pass, but note it in overall_issues.
"""

# import json

# validator_data = {
#     "version": "1.0",
#     "description": "Validator prompt template for evaluating LALM tool-use Q/A pair quality.",
#     "placeholders": {
#         "question":   "The generated question to be evaluated (string)",
#         "reasoning":  "The model's stated reasoning before tool calls (string)",
#         "tool_chain": "JSON array of turns, each containing tool_call and tool_result (array)",
#         "n":          "Expected number of tool calls as specified during generation (int)"
#     },
#     "prompt": validator_prompt_template
# }

# print(json.dumps(validator_data, ensure_ascii=False, indent=2))

In [ ]:
tool_chain = [
    {
        "turn": t["turn"],
        "tool_call": t["tool_call"],
        "tool_result": t["tool_result"]
    }
    for t in qa_pair["answer"]["turns"]
]

filled = validator_prompt_template.format(
    question=qa_pair["question"],
    reasoning=qa_pair["answer"]["reasoning"],
    tool_chain=json.dumps(tool_chain, ensure_ascii=False, indent=2),
    n=2
)


In [ ]:
print(filled)